# Temporal and spatial demand patterns

Explore January demand by hour, weekday/weekend, and pickup zone.


In [ ]:
from pathlib import Path
import sys

# Run from either the project root or the notebooks directory.
project_root = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
if str(project_root / "notebooks") not in sys.path:
    sys.path.insert(0, str(project_root / "notebooks"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_01_aggreg.parquet"))


## Total pickups per hour in January


In [ ]:
time = df.tpep_pickup_datetime.unique()

# Sum pickups across all zones for each hour.
hourly_pickups = (
    df.groupby("tpep_pickup_datetime")["trip_count"]
    .sum()
    .sort_index()
)

# Monday–Friday are weekdays; Saturday–Sunday are weekends.
is_weekend = hourly_pickups.index.dayofweek >= 5
weekday_pickups = hourly_pickups[~is_weekend]
weekend_pickups = hourly_pickups[is_weekend]


In [ ]:
fig, ax = plt.subplots(figsize = (12, 4))

# Keep a continuous timeline, colouring each segment by its starting hour.
# Include the next point at each boundary so the line has no gaps.
segment_starts = np.r_[0, np.flatnonzero(is_weekend[1:] != is_weekend[:-1]) + 1]
segment_ends = np.r_[segment_starts[1:], len(hourly_pickups)]
for start, end in zip(segment_starts, segment_ends):
    stop = min(end + 1, len(hourly_pickups))
    color = "tab:blue" if is_weekend[start] else "tab:orange"
    ax.plot(hourly_pickups.index[start:stop], hourly_pickups.values[start:stop], color = color)

from matplotlib.lines import Line2D
ax.legend(handles = [
    Line2D([], [], color = "tab:orange", label = "Weekdays"),
    Line2D([], [], color = "tab:blue", label = "Weekends"),
])
ax.set_xlabel("Hours")
ax.set_ylabel("Pick ups")
fig.autofmt_xdate()
plt.show()


In [ ]:
y_z = []
for z in df.PULocationID.unique():
    y_z.append(df.trip_count[df["PULocationID"] == z].sum())


In [ ]:
X_z = df.PULocationID.unique()
plt.bar(X_z, y_z)
plt.xlabel("Zones")
plt.ylabel("Pick ups")
plt.show()


#### Comparison of hourly demands between weekends and weekdays


In [ ]:
time_pickup = df["tpep_pickup_datetime"]
trip_count = df.trip_count
def get_hour_avg_weekdays(h):
    m = df["trip_count"][(df["tpep_pickup_datetime"].dt.hour == h) & (df["tpep_pickup_datetime"].dt.dayofweek <= 4)].mean()
    return m

def get_hour_avg_weekdends(h):
    m = df["trip_count"][(df["tpep_pickup_datetime"].dt.hour == h) & (df["tpep_pickup_datetime"].dt.dayofweek > 4)].mean()
    return m


In [ ]:
X_h = [x for x in range(24)]
y1 = [get_hour_avg_weekdends(h) for h in X_h]
y2 = [get_hour_avg_weekdays(h) for h in X_h]
plt.plot(X_h, y1, color = "tab:blue", label = "Weekends")
plt.plot(X_h, y2, color = "tab:orange", label = "Weekdays")
plt.xlabel("Hours")
plt.ylabel("Average pickups per zone per hour")
plt.legend()
plt.show()


In [ ]:
y_z_sort = sorted(y_z)


In [ ]:
ten_busiest_zones = y_z_sort[::-1][:10]


In [ ]:
prop_10 = [ten_busiest_zones[i]/df.trip_count.sum() for i in range(len(ten_busiest_zones))]
prop_10


In [ ]:
def get_no_pickups(h):
    s = 0
    for z in df.PULocationID.unique():
        s += (df.trip_count[(df.tpep_pickup_datetime.dt.hour == h) & (df.PULocationID == z)& (df.trip_count == 0)].count())
    return s


In [ ]:
y_n = [get_no_pickups(x) for x in X_h]
plt.plot(X_h, y_n)
plt.xlabel("Hours")
plt.ylabel("No Pickups")
plt.show()
